**only needed to be ran once just to add all the great files to a new folder**

In [4]:
import pandas as pd
import os
import shutil
from glob import glob

# === Paths ===
#excel_file = '/Volumes/External/TJ_SAR/analysis/Master.xlsx'
#shapefile_dir = '/Volumes/External/TJ_SAR/06_shapefiles/all'
#output_dir = '/Volumes/External/TJ_SAR/06_shapefiles/great'  # You can change this

# Create output folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# === Step 1: Read Excel file ===
df = pd.read_excel(excel_file)
filenames = df['original_filename'].dropna().unique()

# === Step 2: Process and copy files ===
for orig_name in filenames:
    if not orig_name.endswith('_pre_overlay.png'):
        print(f"⚠️ Skipping non-matching filename: {orig_name}")
        continue

    # Replace suffix to get shapefile base name
    shapefile_base = orig_name.replace('_pre_overlay.png', '_pre_mask')
    matching_files = glob(os.path.join(shapefile_dir, f"{shapefile_base}.*"))

    if not matching_files:
        print(f"⚠️ No shapefile components found for {shapefile_base}")
        continue

    for file in matching_files:
        try:
            shutil.copy(file, output_dir)
        except Exception as e:
            print(f"❌ Error copying {file}: {e}")

print("✅ Done copying all matching shapefiles.")


⚠️ No shapefile components found for S1A_IW_GRDH_1SDV_20240825T134455_20240825T134520_055370_06C0B3_2887_pre_mask
⚠️ No shapefile components found for S1A_IW_GRDH_1SDV_20240403T134457_20240403T134522_053270_0674E8_A9EF_pre_mask
⚠️ No shapefile components found for S1A_IW_GRDH_1SDV_20240303T015016_20240303T015045_052811_066417_AE29_pre_mask
⚠️ No shapefile components found for S1A_IW_GRDH_1SDV_20230917T015020_20230917T015049_050361_06103C_F092_pre_mask
✅ Done copying all matching shapefiles.


In [5]:
import geopandas as gpd
import os
from shapely.geometry import Polygon
from shapely.ops import unary_union
import numpy as np
from collections import Counter
import os
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Paths
obs_path = "/Volumes/External/TJ_SAR/01_data/shapefiles/plumeExtentv4.shp"
pred_base_path = "/Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea"

# Load observed shapefile
print(f"Loading observed shapefile from:\n  {obs_path}")
obs_gdf = gpd.read_file(obs_path)
print(f"Total observed features: {len(obs_gdf)}")

# Store IoUs and missing info
ious = []
missing = []

# Iterate through observed features
for idx, row in obs_gdf.iterrows():
    try:
        name = f"{row['sourceName']}_pre_mask_algo_{row['outflow']}"
    except KeyError as e:
        print(f"\n⚠ Missing key column in observed data: {e}")
        missing.append((f"row_{idx}", "observed (missing key)"))
        continue

    pred_path = os.path.join(pred_base_path, f"{name}.shp")

    print(f"\nProcessing feature {idx + 1}/{len(obs_gdf)}:")
    print(f"  Expected predicted shapefile: {pred_path}")

    # Check observed geometry
    if not row.geometry or row.geometry.is_empty:
        print(f"  ❌ Missing or empty observed geometry for: {name}")
        missing.append((name, "observed (empty geometry)"))
        continue

    if os.path.exists(pred_path):
        print(f"  ✔ Found predicted shapefile.")

        try:
            pred_gdf = gpd.read_file(pred_path)

            # Validate geometries
            obs_geom = row.geometry
            pred_geoms = pred_gdf.geometry

            if not obs_geom.is_valid:
                print(f"  ⚠ Observed geometry invalid. Attempting unary_union for cleanup.")
            obs_geom = unary_union([obs_geom])

            valid_pred_geoms = [geom for geom in pred_geoms if geom.is_valid]
            if not valid_pred_geoms:
                print(f"  ⚠ All predicted geometries invalid or empty. Skipping.")
                missing.append((name, "predicted (empty or invalid geometry)"))
                continue

            pred_geom = unary_union(valid_pred_geoms)

            # Compute intersection and union
            intersection = obs_geom.intersection(pred_geom).area
            union = obs_geom.union(pred_geom).area

            print(f"  Intersection area: {intersection:.4f}")
            print(f"  Union area: {union:.4f}")

            if union > 0:
                iou = intersection / union
                ious.append(iou)
                print(f"  ✅ IoU: {iou:.4f}")
            else:
                print(f"  ⚠ Union area is zero. Skipping IoU computation.")
                missing.append((name, "geometry (zero union)"))
        except Exception as e:
            print(f"  ❌ Error reading predicted shapefile for {name}: {e}")
            missing.append((name, "predicted (read error)"))
    else:
        print(f"  ❌ Predicted shapefile not found for: {name}")
        missing.append((name, "predicted (not found)"))

# Output summary
print("\n========== SUMMARY ==========")
if ious:
    print(f"✔ IoU computed for {len(ious)} feature pairs.")
    print(f"Mean IoU: {np.mean(ious):.4f}")
    print(f"Median IoU: {np.median(ious):.4f}")
else:
    print("❌ No IoUs computed.")

# Summarize missing cases
if missing:
    counts = Counter([m[1] for m in missing])
    print(f"\nMissing analysis ({len(missing)} total):")
    for reason, count in counts.items():
        print(f"  - {reason}: {count} features")

    print("\nList of missing or skipped cases:")
    for name, reason in missing:
        print(f"  - {name} ({reason})")


Loading observed shapefile from:
  /Volumes/External/TJ_SAR/01_data/shapefiles/plumeExtentv4.shp
Total observed features: 65

Processing feature 1/65:
  Expected predicted shapefile: /Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea/S1A_IW_GRDH_1SDV_20241216T015014_20241216T015043_057011_070161_E8F2_pre_mask_algo_PB.shp
  ✔ Found predicted shapefile.
  Intersection area: 418105.5367
  Union area: 2219405.5808
  ✅ IoU: 0.1884

Processing feature 2/65:
  Expected predicted shapefile: /Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea/S1A_IW_GRDH_1SDV_20240930T134456_20240930T134521_055895_06D520_6A0F_pre_mask_algo_PB.shp
  ❌ Predicted shapefile not found for: S1A_IW_GRDH_1SDV_20240930T134456_20240930T134521_055895_06D520_6A0F_pre_mask_algo_PB

Processing feature 3/65:
  Expected predicted shapefile: /Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea/S1A_IW_GRDH_1SDV_20240923T015016_20240923T015045_055786_06D0CD_4C0A_pre_mask_algo_PB.shp
  

In [6]:
from pathlib import Path  # ✅ Add this line
# Updated path to observed shapefile
obs_path = "/Volumes/External/TJ_SAR/01_data/shapefiles/plumeExtentv4.shp"
pred_base_path = "/Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea"

# Load observed shapefile
obs_gdf = gpd.read_file(obs_path)

# Store IoUs and metadata
results = []

# Iterate through observed features
for idx, row in obs_gdf.iterrows():
    name = f"{row['sourceName']}_pre_mask_algo_{row['outflow']}"
    pred_path = os.path.join(pred_base_path, f"{name}.shp")

    if os.path.exists(pred_path):
        try:
            pred_gdf = gpd.read_file(pred_path)
            obs_geom = unary_union([row.geometry])
            pred_geom = unary_union([geom for geom in pred_gdf.geometry if geom.is_valid])

            intersection = obs_geom.intersection(pred_geom).area
            union = obs_geom.union(pred_geom).area
            iou = intersection / union if union > 0 else 0

            results.append({
                "index": idx + 1,
                "name": name,
                "obs_geom": obs_geom,
                "pred_geom": pred_geom,
                "iou": iou
            })
        except Exception as e:
            continue

# Create plots
figures = []
output_dir = "/Volumes/External/TJ_SAR/05_viz/IoUplots"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Save IoU overlay plots
for result in results:
    import matplotlib.patches as mpatches

    # Inside your loop:
    fig, ax = plt.subplots()
    gpd.GeoSeries(result["obs_geom"]).plot(ax=ax, color="blue", alpha=0.5)
    gpd.GeoSeries(result["pred_geom"]).plot(ax=ax, color="red", alpha=0.5)

    # Custom legend handles
    obs_patch = mpatches.Patch(color='blue', alpha=0.5, label='Observed')
    pred_patch = mpatches.Patch(color='red', alpha=0.5, label='Predicted')
    plt.legend(handles=[obs_patch, pred_patch])

    plt.title(f"{result['name']}\nIoU: {result['iou']:.4f}")
    plot_path = os.path.join(output_dir, f"{result['index']:02d}_{result['name']}_iou.png")
    plt.savefig(plot_path)
    plt.close()


# Create DataFrame
iou_df = pd.DataFrame(results)[["index", "name", "iou"]]
iou_df.sort_values(by="index", inplace=True)

# Drop NaN values and compute statistics
valid_ious = iou_df["iou"].dropna()
mean_iou = valid_ious.mean()
median_iou = valid_ious.median()

# Print stats
print(f"\nMean IoU: {mean_iou:.4f}")
print(f"Median IoU: {median_iou:.4f}")

# Output CSV
output_csv_path = os.path.join(output_dir, "iou_summary.csv")
iou_df.to_csv(output_csv_path, index=False)
print(f"IoU results written to: {output_csv_path}")




Mean IoU: 0.3630
Median IoU: 0.3401
IoU results written to: /Volumes/External/TJ_SAR/05_viz/IoUplots/iou_summary.csv


In [2]:
import geopandas as gpd
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path
import matplotlib.patches as mpatches

# === Input Paths ===
obs_path = "/Volumes/External/TJ_SAR/01_data/shapefiles/plumeExtentv4.shp"
pred_base_path = "/Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea"

# === Output Directory ===
output_dir = "/Volumes/External/TJ_SAR/05_viz/IoUplots"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# === Load Observed Shapefile and Reproject ===
obs_gdf = gpd.read_file(obs_path)
obs_gdf = obs_gdf.to_crs(epsg=32611)  # UTM Zone 11N

# === Store Results ===
results = []

# === Loop Through Observed Features ===
for idx, row in obs_gdf.iterrows():
    name = f"{row['sourceName']}_pre_mask_algo_{row['outflow']}"
    pred_path = os.path.join(pred_base_path, f"{name}.shp")

    if os.path.exists(pred_path):
        try:
            # Load and reproject predicted shapefile
            pred_gdf = gpd.read_file(pred_path)
            pred_gdf = pred_gdf.to_crs(epsg=32611)

            # Prepare geometries
            obs_geom = unary_union([row.geometry])
            pred_geom = unary_union([geom for geom in pred_gdf.geometry if geom.is_valid])

            # IoU calculation
            intersection = obs_geom.intersection(pred_geom).area
            union = obs_geom.union(pred_geom).area
            iou = intersection / union if union > 0 else 0

            # Area in square kilometers
            obs_area_km2 = obs_geom.area / 1e6
            pred_area_km2 = pred_geom.area / 1e6

            # Cleaned name
            clean_name = name.replace("_pre_mask_algo_", "_")

            # Append result
            results.append({
                "index": idx + 1,
                "name": name,
                "clean_name": clean_name,
                "obs_geom": obs_geom,
                "pred_geom": pred_geom,
                "iou": iou,
                "obs_area_km2": obs_area_km2,
                "pred_area_km2": pred_area_km2
            })

        except Exception as e:
            print(f"Error processing {name}: {e}")
            continue

# === Generate Overlay Plots ===
for result in results:
    fig, ax = plt.subplots()
    gpd.GeoSeries(result["obs_geom"]).plot(ax=ax, color="blue", alpha=0.5)
    gpd.GeoSeries(result["pred_geom"]).plot(ax=ax, color="red", alpha=0.5)

    # Legend
    obs_patch = mpatches.Patch(color='blue', alpha=0.5, label='Observed')
    pred_patch = mpatches.Patch(color='red', alpha=0.5, label='Predicted')
    plt.legend(handles=[obs_patch, pred_patch])

    # Title and save
    plt.title(f"{result['name']}\nIoU: {result['iou']:.4f}")
    plot_path = os.path.join(output_dir, f"{result['index']:02d}_{result['name']}_iou.png")
    plt.savefig(plot_path)
    plt.close()

# === Create DataFrame and Export CSV ===
iou_df = pd.DataFrame(results)[[
    "index", "name", "clean_name", "iou", "obs_area_km2", "pred_area_km2"
]]
iou_df.sort_values(by="index", inplace=True)

# Stats
valid_ious = iou_df["iou"].dropna()
mean_iou = valid_ious.mean()
median_iou = valid_ious.median()

print(f"\nMean IoU: {mean_iou:.4f}")
print(f"Median IoU: {median_iou:.4f}")

# Export CSV
output_csv_path = os.path.join(output_dir, "iou_summary.csv")
iou_df.to_csv(output_csv_path, index=False)
print(f"IoU results written to: {output_csv_path}")




Mean IoU: 0.3630
Median IoU: 0.3401
IoU results written to: /Volumes/External/TJ_SAR/05_viz/IoUplots/iou_summary.csv
